# Phase 1: UsnJrnl Raw Data Parsing

## Overview
This notebook parses the NTFS Update Sequence Number Journal ($UsnJrnl:$J) to extract file change events for timestomping detection analysis.

### Purpose
Extract the following columns from $UsnJrnl:
- **USN**: Update Sequence Number (record identifier)
- **FRN**: File Reference Number (MFT entry number)
- **Parent FRN**: Parent directory FRN
- **Timestamp**: When the change event occurred
- **File Name**: Name of the file
- **Reason Flags**: Type of change (e.g., FILE_CREATE, BASIC_INFO_CHANGE, CLOSE)
- **Reason Codes**: Human-readable reason descriptions
- **Source Info**: Source of the change

### Detection Relevance
The UsnJrnl is critical for detecting timestamp manipulation because:
1. **BASIC_INFO_CHANGE** flag indicates timestamp or attribute modifications
2. Cross-referencing with $SI-E in MFT can reveal discrepancies
3. File System Tunneling patterns can be identified
4. Sequencing of events helps detect manipulation patterns

### Input
- Raw $UsnJrnl file: `data/raw/PE/01-PE/$UsnJrnl_$J.bin`

### Output
- Parsed CSV: `data/Phase 1: Raw Data Parsing/01-PE/UsnJrnl.csv`

### Library
Uses [dfir_ntfs](https://github.com/bamonskiy-kaban/dfir_ntfs) for NTFS artifact parsing.

---
**Reference**: Oh, Lee, and Hwang (2024) - Algorithms 5-7 (UsnJrnl-1A)


In [1]:
# [Cell 1] Install and Import Dependencies
# Install dfir_ntfs if not already installed
import subprocess
import sys

def install_package(package):
    """Install a package using pip."""
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

# Install dfir_ntfs from GitHub
try:
    import dfir_ntfs
    print(f"dfir_ntfs version: {dfir_ntfs.__version__}")
except ImportError:
    print("Installing dfir_ntfs...")
    install_package("git+https://github.com/bamonskiy-kaban/dfir_ntfs.git")
    import dfir_ntfs
    print(f"dfir_ntfs installed successfully. Version: {dfir_ntfs.__version__}")


dfir_ntfs version: 1.1.19


In [2]:
# [Cell 2] Import Required Libraries
import os
import pandas as pd
from datetime import datetime
from pathlib import Path

# dfir_ntfs modules
from dfir_ntfs.USN import ChangeJournalParser, ResolveReasonCodes

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

print("Libraries imported successfully.")


Libraries imported successfully.


In [3]:
# [Cell 3] Define Paths and Configuration

# Base project directory
PROJECT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis")

# Input path - raw UsnJrnl file
INPUT_USNJRNL = PROJECT_DIR / "data" / "raw" / "PE" / "01-PE" / "$UsnJrnl_$J.bin"

# Output directory
OUTPUT_DIR = PROJECT_DIR / "data" / "Phase 1: Raw Data Parsing" / "01-PE"
OUTPUT_CSV = OUTPUT_DIR / "UsnJrnl.csv"

# Create output directory if it does not exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify input file exists
if INPUT_USNJRNL.exists():
    file_size_mb = INPUT_USNJRNL.stat().st_size / (1024 * 1024)
    print(f"Input UsnJrnl file: {INPUT_USNJRNL}")
    print(f"File size: {file_size_mb:.2f} MB")
else:
    raise FileNotFoundError(f"UsnJrnl file not found: {INPUT_USNJRNL}")

print(f"Output will be saved to: {OUTPUT_CSV}")


Input UsnJrnl file: /Users/soni/Github/Digital-Detectives_Thesis/data/raw/PE/01-PE/$UsnJrnl_$J.bin
File size: 34.59 MB
Output will be saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 1: Raw Data Parsing/01-PE/UsnJrnl.csv


In [4]:
# [Cell 3] Define Paths and Configuration

# Base project directory
PROJECT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis")

# Input path - raw UsnJrnl file
INPUT_USNJRNL = PROJECT_DIR / "data" / "raw" / "PE" / "01-PE" / "$UsnJrnl_$J.bin"

# Output directory
OUTPUT_DIR = PROJECT_DIR / "data" / "Phase 1: Raw Data Parsing" / "01-PE"
OUTPUT_CSV = OUTPUT_DIR / "UsnJrnl.csv"

# Create output directory if it does not exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify input file exists
if INPUT_USNJRNL.exists():
    file_size_mb = INPUT_USNJRNL.stat().st_size / (1024 * 1024)
    print(f"Input UsnJrnl file: {INPUT_USNJRNL}")
    print(f"File size: {file_size_mb:.2f} MB")
else:
    raise FileNotFoundError(f"UsnJrnl file not found: {INPUT_USNJRNL}")

print(f"Output will be saved to: {OUTPUT_CSV}")


Input UsnJrnl file: /Users/soni/Github/Digital-Detectives_Thesis/data/raw/PE/01-PE/$UsnJrnl_$J.bin
File size: 34.59 MB
Output will be saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 1: Raw Data Parsing/01-PE/UsnJrnl.csv


In [5]:
# [Cell 4] Define Reason Flag Constants and Helper Functions

# Define critical reason flags for detection
REASON_FLAGS = {
    0x00000001: "DATA_OVERWRITE",
    0x00000002: "DATA_EXTEND",
    0x00000004: "DATA_TRUNCATION",
    0x00000010: "NAMED_DATA_OVERWRITE",
    0x00000020: "NAMED_DATA_EXTEND",
    0x00000040: "NAMED_DATA_TRUNCATION",
    0x00000100: "FILE_CREATE",
    0x00000200: "FILE_DELETE",
    0x00001000: "EA_CHANGE",
    0x00002000: "SECURITY_CHANGE",
    0x00004000: "RENAME_OLD_NAME",
    0x00008000: "RENAME_NEW_NAME",
    0x00010000: "INDEXABLE_CHANGE",
    0x00020000: "BASIC_INFO_CHANGE",
    0x00040000: "HARD_LINK_CHANGE",
    0x00080000: "COMPRESSION_CHANGE",
    0x00100000: "ENCRYPTION_CHANGE",
    0x00200000: "OBJECT_ID_CHANGE",
    0x00400000: "REPARSE_POINT_CHANGE",
    0x00800000: "STREAM_CHANGE",
    0x80000000: "CLOSE"
}

def format_timestamp(dt_obj):
    """
    Format datetime object to string with microsecond precision.
    Returns None if the timestamp is invalid.
    
    Args:
        dt_obj: datetime object or None
        
    Returns:
        str: ISO format timestamp string or None
    """
    if dt_obj is None:
        return None
    try:
        # Format with full precision (microseconds)
        return dt_obj.strftime("%Y-%m-%d %H:%M:%S.%f")
    except (ValueError, AttributeError):
        return None

def parse_reason_flags(reason_code):
    """
    Parse reason code integer into list of flag names.
    
    Args:
        reason_code: Integer bitmask of reason flags
        
    Returns:
        str: Pipe-separated list of flag names
    """
    if reason_code == 0:
        return "NONE"
    
    flags = []
    for mask, name in REASON_FLAGS.items():
        if reason_code & mask:
            flags.append(name)
    
    return " | ".join(flags) if flags else f"UNKNOWN_0x{reason_code:08X}"

def check_basic_detection_pattern(reason_code):
    """
    Check if the reason code contains BASIC_INFO_CHANGE flag.
    This is a key indicator for timestamp manipulation (Algorithm 5).
    
    Args:
        reason_code: Integer bitmask of reason flags
        
    Returns:
        bool: True if BASIC_INFO_CHANGE is present
    """
    return bool(reason_code & 0x00020000)

def check_close_pattern(reason_code):
    """
    Check if the reason code contains CLOSE flag.
    Used in conjunction with BASIC_INFO_CHANGE for detection.
    
    Args:
        reason_code: Integer bitmask of reason flags
        
    Returns:
        bool: True if CLOSE is present
    """
    return bool(reason_code & 0x80000000)

def check_file_create_pattern(reason_code):
    """
    Check if the reason code contains FILE_CREATE flag.
    Used for Algorithm 6 detection.
    
    Args:
        reason_code: Integer bitmask of reason flags
        
    Returns:
        bool: True if FILE_CREATE is present
    """
    return bool(reason_code & 0x00000100)

print("Helper functions and constants defined successfully.")


Helper functions and constants defined successfully.


In [6]:
# [Cell 5] Parse UsnJrnl and Extract Records

def parse_usnjrnl(usnjrnl_path, progress_interval=10000):
    """
    Parse the UsnJrnl file and extract all relevant metadata.
    
    Args:
        usnjrnl_path: Path to the $UsnJrnl:$J file
        progress_interval: Print progress every N records
        
    Returns:
        list: List of dictionaries containing parsed records
    """
    records = []
    
    print(f"Opening UsnJrnl file: {usnjrnl_path}")
    
    with open(usnjrnl_path, "rb") as usn_file:
        parser = ChangeJournalParser(usn_file)
        
        record_count = 0
        error_count = 0
        
        print("Parsing UsnJrnl records...")
        
        for usn_record in parser.usn_records():
            try:
                record_count += 1
                
                # Progress indicator
                if record_count % progress_interval == 0:
                    print(f"  Processed {record_count:,} records...")
                
                # Extract record data
                usn = usn_record.get_usn()
                frn = usn_record.get_file_reference_number() & 0xFFFFFFFFFFFF  # Lower 48 bits
                parent_frn = usn_record.get_parent_file_reference_number() & 0xFFFFFFFFFFFF
                timestamp = format_timestamp(usn_record.get_timestamp())
                filename = usn_record.get_file_name()
                reason_code = usn_record.get_reason()
                reason_flags = parse_reason_flags(reason_code)
                source_info = usn_record.get_source_info()
                
                # Detection pattern flags
                has_basic_info_change = check_basic_detection_pattern(reason_code)
                has_close = check_close_pattern(reason_code)
                has_file_create = check_file_create_pattern(reason_code)
                
                # Combine all info into a record
                record = {
                    "USN": usn,
                    "FRN": frn,
                    "ParentFRN": parent_frn,
                    "Timestamp": timestamp,
                    "FileName": filename,
                    "ReasonCode": reason_code,
                    "ReasonFlags": reason_flags,
                    "SourceInfo": source_info,
                    "HasBasicInfoChange": has_basic_info_change,
                    "HasClose": has_close,
                    "HasFileCreate": has_file_create
                }
                
                records.append(record)
                
            except Exception as e:
                error_count += 1
                if error_count <= 5:
                    print(f"  Warning: Error parsing record {record_count}: {str(e)[:100]}")
                elif error_count == 6:
                    print("  (Suppressing further error messages...)")
    
    print(f"\nParsing complete!")
    print(f"  Total records processed: {record_count:,}")
    print(f"  Records with errors: {error_count:,}")
    print(f"  Successfully parsed: {len(records):,}")
    
    return records

# Execute parsing
usn_records = parse_usnjrnl(INPUT_USNJRNL)


Opening UsnJrnl file: /Users/soni/Github/Digital-Detectives_Thesis/data/raw/PE/01-PE/$UsnJrnl_$J.bin
Parsing UsnJrnl records...
  Processed 10,000 records...
  Processed 20,000 records...
  Processed 30,000 records...
  Processed 40,000 records...
  Processed 50,000 records...
  Processed 60,000 records...
  Processed 70,000 records...
  Processed 80,000 records...
  Processed 90,000 records...
  Processed 100,000 records...
  Processed 110,000 records...
  Processed 120,000 records...
  Processed 130,000 records...
  Processed 140,000 records...
  Processed 150,000 records...
  Processed 160,000 records...
  Processed 170,000 records...
  Processed 180,000 records...
  Processed 190,000 records...
  Processed 200,000 records...
  Processed 210,000 records...
  Processed 220,000 records...
  Processed 230,000 records...
  Processed 240,000 records...
  Processed 250,000 records...
  Processed 260,000 records...
  Processed 270,000 records...
  Processed 280,000 records...
  Processed 2

In [7]:
# [Cell 6] Create DataFrame and Organize Columns

# Create DataFrame
df_usn = pd.DataFrame(usn_records)

# Define column order (matching the required output format)
column_order = [
    "USN",                    # Update Sequence Number
    "FRN",                    # File Reference Number
    "ParentFRN",              # Parent Directory FRN
    "Timestamp",              # Event timestamp
    "FileName",               # File name
    "ReasonCode",             # Integer reason code
    "ReasonFlags",            # Human-readable flags
    "SourceInfo",             # Source of change
    "HasBasicInfoChange",     # Detection flag
    "HasClose",               # Detection flag
    "HasFileCreate"           # Detection flag
]

# Reorder columns
df_usn = df_usn[column_order]

print(f"DataFrame created with {len(df_usn):,} records and {len(df_usn.columns)} columns")
print(f"\nColumn names: {list(df_usn.columns)}")


DataFrame created with 316,817 records and 11 columns

Column names: ['USN', 'FRN', 'ParentFRN', 'Timestamp', 'FileName', 'ReasonCode', 'ReasonFlags', 'SourceInfo', 'HasBasicInfoChange', 'HasClose', 'HasFileCreate']


In [8]:
# [Cell 7] Data Quality Summary

print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

# Basic statistics
print(f"\nTotal Records: {len(df_usn):,}")

# Missing values
print("\nMissing Values:")
for col in df_usn.columns:
    null_count = df_usn[col].isnull().sum()
    null_pct = (null_count / len(df_usn)) * 100
    print(f"  {col}: {null_count:,} ({null_pct:.2f}%)")

# Detection pattern statistics
print("\n" + "=" * 60)
print("DETECTION PATTERN STATISTICS")
print("=" * 60)

basic_info_count = df_usn["HasBasicInfoChange"].sum()
close_count = df_usn["HasClose"].sum()
file_create_count = df_usn["HasFileCreate"].sum()

print(f"\nRecords with BASIC_INFO_CHANGE: {basic_info_count:,}")
print(f"Records with CLOSE: {close_count:,}")
print(f"Records with FILE_CREATE: {file_create_count:,}")

# Check for basic detection pattern (BASIC_INFO_CHANGE + CLOSE)
both_flags = df_usn[df_usn["HasBasicInfoChange"] & df_usn["HasClose"]]
print(f"Records with BASIC_INFO_CHANGE + CLOSE: {len(both_flags):,}")

# Sample data
print("\n" + "=" * 60)
print("SAMPLE RECORDS (First 5)")
print("=" * 60)
df_usn.head()


DATA QUALITY SUMMARY

Total Records: 316,817

Missing Values:
  USN: 0 (0.00%)
  FRN: 0 (0.00%)
  ParentFRN: 0 (0.00%)
  Timestamp: 0 (0.00%)
  FileName: 0 (0.00%)
  ReasonCode: 0 (0.00%)
  ReasonFlags: 0 (0.00%)
  SourceInfo: 0 (0.00%)
  HasBasicInfoChange: 0 (0.00%)
  HasClose: 0 (0.00%)
  HasFileCreate: 0 (0.00%)

DETECTION PATTERN STATISTICS

Records with BASIC_INFO_CHANGE: 368
Records with CLOSE: 146,395
Records with FILE_CREATE: 153,995
Records with BASIC_INFO_CHANGE + CLOSE: 93

SAMPLE RECORDS (First 5)


,USN,FRN,ParentFRN,Timestamp,FileName,ReasonCode,ReasonFlags,SourceInfo,HasBasicInfoChange,HasClose,HasFileCreate
0,1291845632,518229,344827,2023-12-19 07:12:25.736406,dsreg.dll.mui,2147483648,CLOSE,0,False,True,False
1,1291845720,518230,344827,2023-12-19 07:12:25.736406,dsregcmd.exe.mui,258,DATA_EXTEND | FILE_CREATE,0,False,False,True
2,1291845816,518230,344827,2023-12-19 07:12:25.736406,dsregcmd.exe.mui,1048834,DATA_EXTEND | FILE_CREATE | ENCRYPTION_CHANGE,0,False,False,True
3,1291845912,518230,344827,2023-12-19 07:12:25.736406,dsregcmd.exe.mui,3145986,DATA_EXTEND | FILE_CREATE | ENCRYPTION_CHANGE | OBJECT_ID_CHANGE,0,False,False,True
4,1291846008,518230,344827,2023-12-19 07:12:25.736406,dsregcmd.exe.mui,3146018,DATA_EXTEND | NAMED_DATA_EXTEND | FILE_CREATE | ENCRYPTION_CHANGE | OBJECT_ID_CHANGE,0,False,False,True


In [9]:
# [Cell 8] Analyze Most Common Reason Flags

print("=" * 60)
print("TOP 20 MOST COMMON REASON FLAGS")
print("=" * 60)

# Count occurrences of each reason flag combination
reason_counts = df_usn["ReasonFlags"].value_counts().head(20)

print("\nRank | Count     | Reason Flags")
print("-" * 60)
for idx, (reason, count) in enumerate(reason_counts.items(), 1):
    pct = (count / len(df_usn)) * 100
    print(f"{idx:3d}  | {count:8,} ({pct:5.2f}%) | {reason[:80]}")

# Check for timestamp manipulation indicators
print("\n" + "=" * 60)
print("TIMESTAMP MANIPULATION INDICATORS")
print("=" * 60)

# Pattern 1: BASIC_INFO_CHANGE alone
basic_only = df_usn[df_usn["ReasonFlags"].str.contains("BASIC_INFO_CHANGE", na=False)]
print(f"\nRecords with BASIC_INFO_CHANGE flag: {len(basic_only):,}")

# Pattern 2: BASIC_INFO_CHANGE followed by CLOSE (need temporal analysis in Phase 2)
print(f"Records with both BASIC_INFO_CHANGE and CLOSE flags: {len(both_flags):,}")
print("  (Temporal proximity analysis will be performed in Phase 2)")


TOP 20 MOST COMMON REASON FLAGS

Rank | Count     | Reason Flags
------------------------------------------------------------
  1  |   44,367 (14.00%) | FILE_CREATE
  2  |   40,633 (12.83%) | FILE_DELETE | CLOSE
  3  |   34,656 (10.94%) | DATA_EXTEND | FILE_CREATE
  4  |   20,279 ( 6.40%) | CLOSE
  5  |   16,641 ( 5.25%) | DATA_EXTEND | FILE_CREATE | CLOSE
  6  |   15,752 ( 4.97%) | UNKNOWN_0x00000800
  7  |   13,120 ( 4.14%) | DATA_OVERWRITE | DATA_EXTEND | FILE_CREATE
  8  |   11,923 ( 3.76%) | DATA_OVERWRITE | DATA_EXTEND | FILE_CREATE | CLOSE
  9  |   10,699 ( 3.38%) | INDEXABLE_CHANGE | CLOSE
 10  |    8,575 ( 2.71%) | DATA_TRUNCATION
 11  |    8,096 ( 2.56%) | DATA_EXTEND | DATA_TRUNCATION
 12  |    7,935 ( 2.50%) | FILE_CREATE | CLOSE
 13  |    7,429 ( 2.34%) | DATA_EXTEND | DATA_TRUNCATION | CLOSE
 14  |    6,836 ( 2.16%) | DATA_EXTEND
 15  |    6,018 ( 1.90%) | DATA_EXTEND | CLOSE
 16  |    4,886 ( 1.54%) | DATA_OVERWRITE
 17  |    4,250 ( 1.34%) | DATA_OVERWRITE | CLOSE
 18  

In [10]:
# [Cell 9] Temporal Analysis Preview

print("=" * 60)
print("TEMPORAL DISTRIBUTION")
print("=" * 60)

# Convert timestamp to datetime for analysis
df_usn["TimestampDT"] = pd.to_datetime(df_usn["Timestamp"], errors="coerce")

# Get date range
if df_usn["TimestampDT"].notna().any():
    min_time = df_usn["TimestampDT"].min()
    max_time = df_usn["TimestampDT"].max()
    time_span = max_time - min_time
    
    print(f"\nEarliest Event: {min_time}")
    print(f"Latest Event:   {max_time}")
    print(f"Time Span:      {time_span}")
    
    # Events per day
    df_usn["Date"] = df_usn["TimestampDT"].dt.date
    events_per_day = df_usn.groupby("Date").size()
    
    print(f"\nAverage Events per Day: {events_per_day.mean():.0f}")
    print(f"Max Events in a Day:    {events_per_day.max():,} ({events_per_day.idxmax()})")
    print(f"Min Events in a Day:    {events_per_day.min():,} ({events_per_day.idxmin()})")
    
    # Clean up temporary columns
    df_usn.drop(columns=["TimestampDT", "Date"], inplace=True)
else:
    print("No valid timestamps found for temporal analysis.")


TEMPORAL DISTRIBUTION

Earliest Event: 2023-12-19 07:12:25.736406
Latest Event:   2023-12-22 16:23:49.949860
Time Span:      3 days 09:11:24.213454

Average Events per Day: 105606
Max Events in a Day:    241,523 (2023-12-19)
Min Events in a Day:    11,432 (2023-12-21)


In [11]:
# [Cell 10] File Activity Summary

print("=" * 60)
print("FILE ACTIVITY SUMMARY")
print("=" * 60)

# Top files by activity count
file_activity = df_usn["FileName"].value_counts().head(20)

print("\nTop 20 Most Active Files:")
print("Rank | Count     | File Name")
print("-" * 60)
for idx, (filename, count) in enumerate(file_activity.items(), 1):
    print(f"{idx:3d}  | {count:8,} | {filename}")

# Unique files
unique_frns = df_usn["FRN"].nunique()
unique_files = df_usn["FileName"].nunique()

print(f"\nUnique FRNs (files):      {unique_frns:,}")
print(f"Unique File Names:        {unique_files:,}")
print(f"Average Events per File:  {len(df_usn) / unique_frns:.1f}")


FILE ACTIVITY SUMMARY

Top 20 Most Active Files:
Rank | Count     | File Name
------------------------------------------------------------
  1  |   30,310 | UDB-User23847576+RemoteGraph.sql-journal
  2  |   15,054 | UDB-User23847576+LocalStorage.sql-journal
  3  |   13,932 | psi.db-journal
  4  |    5,490 | ngen.log
  5  |    3,552 | AvEmUpdate.log
  6  |    3,292 | segments.gen
  7  |    3,099 | scope_v2.json
  8  |    2,990 | LocalSettingsDB.sql-journal
  9  |    2,784 | ngenlock.dat
 10  |    2,211 | storage.json
 11  |    1,620 | D566D7D7-DCD6-471C-8109-BE0AD33199E3
 12  |    1,574 | Microsoft.ui.xaml.dll.mui
 13  |    1,550 | Microsoft.UI.Xaml.Phone.dll.mui
 14  |    1,469 | setupapi.dev.log
 15  |    1,410 | config.dbx-journal
 16  |      956 | cache.lock
 17  |      888 | LOG
 18  |      794 | Temp
 19  |      731 | settings.dat
 20  |      683 | www.bing[1].xml

Unique FRNs (files):      23,517
Unique File Names:        35,154
Average Events per File:  13.5


In [12]:
# [Cell 11] Save to CSV

# Drop detection helper columns (keep only essential data for Phase 2)
output_columns = [
    "USN",
    "FRN",
    "ParentFRN",
    "Timestamp",
    "FileName",
    "ReasonCode",
    "ReasonFlags",
    "SourceInfo",
    "HasBasicInfoChange",
    "HasClose",
    "HasFileCreate"
]

df_output = df_usn[output_columns].copy()

# Save DataFrame to CSV
df_output.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

# Verify the saved file
saved_size = OUTPUT_CSV.stat().st_size / (1024 * 1024)
print(f"UsnJrnl data saved to: {OUTPUT_CSV}")
print(f"Output file size: {saved_size:.2f} MB")

# Verify by reading back
df_verify = pd.read_csv(OUTPUT_CSV)
print(f"Verification: {len(df_verify):,} records saved successfully")


UsnJrnl data saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 1: Raw Data Parsing/01-PE/UsnJrnl.csv
Output file size: 39.38 MB
Verification: 316,817 records saved successfully
